# Composer Classification Using CNN and LSTM Models
## Final Team Project

**Team Members** *(listed alphabetically by first name)*

- Andrew Blumhardt
- Brian Covington
- Erika Gallegos

**University:** University of San Diego  
**Course:** AAI-511: Neural Networks and Deep Learning  
**Due Date:** August 10, 2026

### Project Overview

This notebook develops and evaluates two deep learning approaches for classifying classical music composers using symbolic MIDI data. The workflow includes data preparation, feature extraction, model development, training, evaluation, and comparison of a Long Short-Term Memory (LSTM) network and a Convolutional Neural Network (CNN). Throughout the notebook, narrative explanations and code comments describe the purpose of each step and the reasoning behind the implementation.


## 1. Introduction

Music composers often develop recognizable stylistic patterns involving melody, rhythm, harmony, note duration, and structure. The purpose of this project is to determine whether deep learning models can learn those patterns and use them to identify the composer of a previously unseen musical score.

The project uses labeled MIDI files representing works from several classical composers. Two deep learning approaches are compared:

- A **Long Short-Term Memory (LSTM)** network that analyzes music as a sequence of notes over time.
- A **Convolutional Neural Network (CNN)** that analyzes a two-dimensional piano-roll representation of each composition.

The primary objective is to compare the models using accuracy, precision, recall, F1 score, and confusion matrices. The completed notebook documents the entire process, including data preparation, feature extraction, model development, training, evaluation, and conclusions.

## 2. Attribution and Use of External Resources

This project was completed collaboratively by the team members listed above. The notebook structure and model-development approach were informed by the TensorFlow, CNN, RNN, and autoencoder examples provided in the course. The team adapted those general examples to the composer-classification problem rather than reproducing them directly.

Generative AI tools may have been used to assist with notebook organization, code review, debugging, and editing. All generated material was reviewed, tested, and revised by the team. The team remains responsible for the accuracy of the code, analysis, and written conclusions.

Specific libraries, frameworks, datasets, and external sources are acknowledged in the **References** section at the end of the notebook.

## 3. Project Workflow

This notebook follows a standard deep learning workflow:

1. Configure the environment and import libraries.
2. Locate and inspect the MIDI dataset.
3. Clean the data and identify composer labels.
4. Extract note-sequence features for the LSTM.
5. Create piano-roll features for the CNN.
6. Split the data into training, validation, and test sets.
7. Build and train both models.
8. Evaluate and compare their performance.
9. Summarize findings, limitations, and future improvements.

## 4. Environment Setup

The following cell installs the MIDI-processing packages used in the project. In Google Colab, the cell normally needs to be run only once per runtime.

`music21` is used to read and analyze MIDI files. TensorFlow/Keras is used to build and train the neural networks.

In [ ]:
# Install the libraries used to read MIDI files and prepare musical features.
# The -q option keeps the Colab output brief by hiding routine installation messages.
!pip install -q music21 pretty_midi


### 4.1 Import Libraries

This cell imports the libraries used throughout the notebook and sets random seeds to make the results more reproducible.

In [ ]:
# Import standard Python utilities for file handling, reproducibility, and summary counts.
import os
import random
import warnings
from pathlib import Path
from collections import Counter

# Import libraries used for numerical processing, tables, and visualizations.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import music21 tools used to parse MIDI files and identify notes, chords, and instruments.
from music21 import converter, instrument, note, chord, stream

# Import scikit-learn tools used to split the data, encode composer labels,
# and calculate classification performance metrics.
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_fscore_support
)

# Import TensorFlow and the Keras layers used to build the LSTM and CNN models.
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input,
    Embedding,
    LSTM,
    Dense,
    Dropout,
    Conv2D,
    MaxPooling2D,
    Flatten,
    BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Suppress noncritical library warnings so the notebook output remains readable.
# This does not suppress Python errors; errors that stop execution will still be displayed.
warnings.filterwarnings("ignore")

# Set a fixed random seed so that data splits and model initialization are as reproducible
# as possible across repeated runs. The number 42 is conventional but arbitrary; any fixed
# integer would serve the same purpose. Changing the seed may slightly change model results.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Display the TensorFlow version because library versions can affect model behavior
# and reproducibility when the notebook is run in a different Colab session.
print("TensorFlow version:", tf.__version__)


### Dataset Upload in Google Colab

The dataset should be compressed into one ZIP archive before running the notebook. Inside the archive, each composer should have a separate folder containing that composer's MIDI files. The following code opens a Colab upload dialog, extracts the archive, and assigns `/content/dataset` as the working dataset path.

Because files uploaded directly to Colab are removed when the runtime ends, Google Drive is a better option when the dataset is large or will be reused across multiple sessions. An alternative Drive-based setup is included in the code comments.


In [ ]:
# Google Colab stores uploaded files only for the current runtime session.
# Upload the dataset as a single ZIP file that contains composer folders and MIDI files.
# Example structure:
# dataset.zip
# ├── Bach/
# │   ├── piece1.mid
# │   └── piece2.mid
# ├── Beethoven/
# └── Mozart/

from google.colab import files
import zipfile

# Open a browser dialog and allow the user to upload the ZIP archive from the local computer.
uploaded = files.upload()

# Identify the uploaded ZIP file. This notebook expects one dataset archive.
zip_files = [name for name in uploaded.keys() if name.lower().endswith(".zip")]

if not zip_files:
    raise ValueError("No ZIP file was uploaded. Upload the MIDI dataset as a .zip archive.")

zip_name = zip_files[0]

# Extract the archive into /content/dataset, which becomes the working dataset location.
# Existing files in this folder may be overwritten if the cell is run more than once.
DATASET_PATH = Path("/content/dataset")
DATASET_PATH.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_name, "r") as zip_ref:
    zip_ref.extractall(DATASET_PATH)

print("Dataset path:", DATASET_PATH)
print("Path exists:", DATASET_PATH.exists())
print("Uploaded archive:", zip_name)

# Alternative for larger datasets:
# Mount Google Drive and point DATASET_PATH to a permanent folder in Drive.
#
# from google.colab import drive
# drive.mount("/content/drive")
# DATASET_PATH = Path("/content/drive/MyDrive/final_project/dataset")


### 5.1 Discover MIDI Files

The following cell searches the dataset recursively for `.mid` and `.midi` files. It assumes that the name of the parent folder represents the composer label.

In [ ]:
# Search the dataset folder and all of its subfolders for common MIDI file extensions.
# rglob is used because the files are expected to be organized in composer subfolders.
midi_files = list(DATASET_PATH.rglob("*.mid")) + list(DATASET_PATH.rglob("*.midi"))

records = []

# Build one metadata record per MIDI file.
# The parent folder name is treated as the composer label, so the folder structure
# directly determines the target class used during model training.
for file_path in midi_files:
    records.append({
        "file_path": str(file_path),
        "file_name": file_path.name,
        "composer": file_path.parent.name
    })

# Convert the records to a DataFrame so the dataset can be inspected and filtered easily.
dataset_df = pd.DataFrame(records)

print(f"Total MIDI files found: {len(dataset_df)}")
dataset_df.head()


### 5.2 Inspect Class Distribution

A class-distribution chart helps determine whether some composers have substantially more files than others. Severe imbalance can cause the models to favor the most common composers.

In [ ]:
# Check the class distribution before training because a large imbalance between composers
# can cause a model to favor classes that have more examples.
if not dataset_df.empty:
    composer_counts = dataset_df["composer"].value_counts().sort_values(ascending=False)
    display(composer_counts.to_frame("file_count"))

    # Plot the number of files per composer to make imbalance easier to identify visually.
    plt.figure(figsize=(10, 5))
    composer_counts.plot(kind="bar")
    plt.title("Number of MIDI Files by Composer")
    plt.xlabel("Composer")
    plt.ylabel("Number of Files")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    # Stop here if no files were found because the remaining cells depend on a valid dataset.
    print("No MIDI files were found. Verify DATASET_PATH before continuing.")


## 6. Data Cleaning and Validation

MIDI collections sometimes contain unreadable, empty, or unusually short files. The next function attempts to parse each file and records whether it can be used.

Files that cannot be parsed are excluded rather than allowing them to stop the entire notebook.

In [ ]:
def validate_midi_file(file_path):
    """Return True when a MIDI file can be parsed and contains notes or chords."""

    try:
        # Attempt to parse the file with music21. Corrupted or unsupported files
        # will raise an exception and should not be included in model training.
        score = converter.parse(file_path)

        # A technically valid MIDI file may still contain no usable musical events.
        # Requiring at least one note or chord prevents empty examples from entering the dataset.
        elements = score.recurse().notes
        return len(elements) > 0

    except Exception:
        # Invalid files are marked False rather than stopping the entire preprocessing process.
        return False


# Validate every file before feature extraction. This step can take several minutes because
# each MIDI file must be opened and parsed individually.
dataset_df["is_valid"] = dataset_df["file_path"].apply(validate_midi_file)

# Separate invalid files for reporting and retain only usable files for later processing.
invalid_files = dataset_df.loc[~dataset_df["is_valid"]]
clean_df = dataset_df.loc[dataset_df["is_valid"]].reset_index(drop=True)

print(f"Valid files: {len(clean_df)}")
print(f"Invalid files removed: {len(invalid_files)}")


## 7. Feature Extraction

The CNN and LSTM require different representations of the same music.

- The **LSTM** receives a sequence of note and chord tokens in the order they occur.
- The **CNN** receives a piano-roll matrix showing pitch activity across time.

Keeping these pipelines separate allows each architecture to work with a representation that fits its strengths.

### 7.1 Extract Note and Chord Sequences for the LSTM

Each note is converted to its MIDI pitch number. Chords are represented as pitch classes joined with periods. For example, a chord containing pitch classes 0, 4, and 7 becomes `0.4.7`.

The resulting sequence preserves the order of musical events.

In [ ]:
def extract_note_tokens(file_path):
    """Extract an ordered sequence of note and chord tokens from one MIDI file."""

    # Parse the MIDI file into a music21 score that can be inspected programmatically.
    score = converter.parse(file_path)

    try:
        # When instrument parts are available, use the first part to reduce duplicated notes
        # that can occur in multi-track MIDI files. If no parts exist, use all flattened notes.
        parts = instrument.partitionByInstrument(score)
        elements = parts.parts[0].recurse() if parts else score.flat.notes
    except Exception:
        # Fall back to the flattened score when instrument partitioning is unavailable.
        elements = score.flat.notes

    tokens = []

    # Preserve the order of notes and chords because the LSTM is designed to learn
    # relationships between musical events over time.
    for element in elements:
        if isinstance(element, note.Note):
            # Represent a single note by its MIDI pitch number.
            tokens.append(str(element.pitch.midi))

        elif isinstance(element, chord.Chord):
            # Represent a chord as pitch classes joined into one token so simultaneous
            # notes are treated as a single musical event.
            chord_token = ".".join(str(n) for n in element.normalOrder)
            tokens.append(chord_token)

    return tokens


### 7.2 Create Piano-Roll Matrices for the CNN

A piano roll is a two-dimensional matrix:

- Rows represent MIDI pitches.
- Columns represent time steps.
- Cell values indicate whether a pitch is active.

The matrices are resized or padded to a consistent shape so they can be processed in batches by the CNN.

In [ ]:
def midi_to_piano_roll(file_path, time_steps=256, pitch_min=21, pitch_max=108):
    """Convert a MIDI file into a fixed-size binary piano-roll matrix."""

    # Parse the MIDI file and create an empty pitch-by-time matrix.
    # The default pitch range 21-108 corresponds to the standard 88-key piano.
    score = converter.parse(file_path)
    matrix = np.zeros((pitch_max - pitch_min + 1, time_steps), dtype=np.float32)

    # Flatten the score so notes and chords from all tracks can be placed on one timeline.
    notes_and_chords = list(score.flat.notes)

    # Return an all-zero matrix if no usable events are present.
    if not notes_and_chords:
        return matrix

    # MIDI files differ in length, but CNN inputs must have identical dimensions.
    # Scale each piece to 256 time positions so every example has the same width.
    max_offset = max(float(element.offset) for element in notes_and_chords)
    scale = (time_steps - 1) / max(max_offset, 1.0)

    for element in notes_and_chords:
        # Convert the original musical offset and duration into matrix positions.
        start = int(float(element.offset) * scale)
        duration = max(1, int(float(element.quarterLength) * scale))
        end = min(time_steps, start + duration)

        if isinstance(element, note.Note):
            pitch = element.pitch.midi

            # Ignore pitches outside the selected piano range so every matrix has
            # the same number of pitch rows.
            if pitch_min <= pitch <= pitch_max:
                matrix[pitch - pitch_min, start:end] = 1.0

        elif isinstance(element, chord.Chord):
            # Mark every pitch in a chord as active across the chord's duration.
            for chord_note in element.notes:
                pitch = chord_note.pitch.midi
                if pitch_min <= pitch <= pitch_max:
                    matrix[pitch - pitch_min, start:end] = 1.0

    return matrix


### 7.3 Extract Features from All Files

Feature extraction is usually the slowest preprocessing step. The code below processes each valid file and skips any file that produces an error.

To reduce repeated processing, the resulting arrays can later be saved to disk with `numpy.save`.

In [ ]:
# Store two different feature representations of each piece:
# an ordered token sequence for the LSTM and a piano-roll matrix for the CNN.
lstm_sequences = []
cnn_matrices = []
labels = []
failed_files = []

# Process every validated MIDI file using both feature extraction methods.
for _, row in clean_df.iterrows():
    try:
        note_sequence = extract_note_tokens(row["file_path"])
        piano_roll = midi_to_piano_roll(row["file_path"])

        # Require at least 20 sequential events so the LSTM has enough musical context
        # to learn a meaningful pattern. Shorter files are excluded from both models
        # so the CNN and LSTM are compared using the same set of pieces.
        if len(note_sequence) >= 20:
            lstm_sequences.append(note_sequence)
            cnn_matrices.append(piano_roll)
            labels.append(row["composer"])

    except Exception as exc:
        # Record failures instead of terminating the full extraction process.
        failed_files.append((row["file_path"], str(exc)))

print(f"Files successfully processed: {len(labels)}")
print(f"Files skipped during feature extraction: {len(failed_files)}")


## 8. Prepare Labels and Data Splits

Composer names are converted into integer class labels. The same train, validation, and test split should be used for both models so the comparison is fair.

A stratified split is used when possible to preserve approximately the same composer distribution in each subset.

In [ ]:
# Convert composer names from text labels into integer class IDs required by Keras.
# The encoder also preserves the mapping so predictions can later be translated back
# into readable composer names.
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels)
class_names = list(label_encoder.classes_)

# Create an index array so the exact same train, validation, and test split can be
# applied to both the LSTM sequences and CNN matrices.
indices = np.arange(len(y))

# Reserve 20% of the complete dataset as a final test set.
# stratify=y keeps the composer proportions similar across the split.
train_idx, test_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

# Split the remaining training portion again to create a validation set.
# A 20% split of the remaining 80% produces approximately 64% training,
# 16% validation, and 20% testing overall.
train_idx, val_idx = train_test_split(
    train_idx,
    test_size=0.20,
    random_state=SEED,
    stratify=y[train_idx]
)

print("Training samples:", len(train_idx))
print("Validation samples:", len(val_idx))
print("Test samples:", len(test_idx))
print("Composer classes:", class_names)


## 9. LSTM Data Preparation

Neural networks require numerical inputs. The note and chord tokens are therefore mapped to integer IDs.

Because compositions contain different numbers of events, each sequence is padded or truncated to the same length. The sequence length can be adjusted after inspecting the dataset.

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Limit each LSTM input to 500 musical events.
# Shorter pieces are padded with zeros, while longer pieces are truncated.
# This fixed length is required so examples can be processed together in batches.
MAX_SEQUENCE_LENGTH = 500

def prepare_lstm_data(sequences, train_indices, val_indices, test_indices):
    """Create a training-only vocabulary and encode sequences for the LSTM."""

    # Build the vocabulary from the training set only.
    # This avoids allowing information from validation or test examples to influence
    # how the input representation is created.
    training_tokens = [
        token
        for index in train_indices
        for token in sequences[index]
    ]

    vocabulary = sorted(set(training_tokens))

    # Reserve ID 0 for padding and unseen tokens. Known musical tokens begin at ID 1.
    token_to_id = {token: idx + 1 for idx, token in enumerate(vocabulary)}

    # Convert note and chord strings into integer IDs that can be passed to an embedding layer.
    # Tokens not observed in training are mapped to 0.
    encoded_sequences = [
        [token_to_id.get(token, 0) for token in sequence]
        for sequence in sequences
    ]

    # Pad or truncate every sequence to the same length.
    # Post-padding preserves the beginning of each sequence in its original order.
    padded = pad_sequences(
        encoded_sequences,
        maxlen=MAX_SEQUENCE_LENGTH,
        padding="post",
        truncating="post"
    )

    return (
        padded[train_indices],
        padded[val_indices],
        padded[test_indices],
        token_to_id
    )


X_lstm_train, X_lstm_val, X_lstm_test, token_to_id = prepare_lstm_data(
    lstm_sequences, train_idx, val_idx, test_idx
)

# Apply the same indices to the encoded composer labels.
y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]


## 10. Build the LSTM Model

The LSTM model uses:

- An embedding layer to learn a compact representation of each note or chord token.
- Two LSTM layers to learn sequential musical patterns.
- Dropout layers to reduce overfitting.
- A softmax output layer to predict the composer.

In [ ]:
def build_lstm_model(vocabulary_size, number_of_classes):
    """Build an LSTM classifier for ordered note and chord sequences."""

    model = Sequential([
        # Each example contains up to 500 integer-encoded musical events.
        Input(shape=(MAX_SEQUENCE_LENGTH,)),

        # Convert token IDs into learned 64-dimensional vectors.
        # mask_zero=True tells the LSTM to ignore padding positions.
        Embedding(
            input_dim=vocabulary_size + 1,
            output_dim=64,
            mask_zero=True
        ),

        # The first LSTM returns a sequence so a second LSTM can continue
        # learning higher-level temporal relationships.
        LSTM(128, return_sequences=True),

        # Dropout randomly disables 30% of units during training to reduce overfitting.
        Dropout(0.30),

        # The second LSTM condenses the sequence into one learned representation.
        LSTM(64),
        Dropout(0.30),

        # A dense layer learns nonlinear combinations of the extracted sequence features.
        Dense(64, activation="relu"),

        # Softmax returns one probability for each composer class.
        Dense(number_of_classes, activation="softmax")
    ])

    # Adam is used because it adapts the learning rate during training.
    # Sparse categorical cross-entropy is appropriate because labels are integer encoded.
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


lstm_model = build_lstm_model(
    vocabulary_size=len(token_to_id),
    number_of_classes=len(class_names)
)

lstm_model.summary()


### 10.1 Train the LSTM

Early stopping ends training when validation loss no longer improves. Reducing the learning rate can help the model make smaller adjustments after progress begins to slow.

In [ ]:
# Use callbacks to limit unnecessary training and respond when validation learning slows.
training_callbacks = [
    EarlyStopping(
        # Validation loss is monitored because it measures performance on unseen data
        # and is often a better signal of overfitting than training accuracy.
        monitor="val_loss",

        # Stop after five consecutive epochs without improvement.
        patience=5,

        # Return the model to the epoch with the lowest validation loss rather than
        # keeping the final, potentially overfit, weights.
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        # Reduce the learning rate when validation loss stops improving.
        monitor="val_loss",

        # Cut the current learning rate in half to allow smaller optimization steps.
        factor=0.5,

        # Wait two stagnant epochs before reducing the learning rate.
        patience=2,

        # Prevent the learning rate from becoming so small that training effectively stops.
        min_lr=1e-6
    )
]

# Train for at most 30 epochs. Early stopping may end training sooner.
# A batch size of 32 provides a practical balance between stable gradient updates
# and Colab memory usage.
lstm_history = lstm_model.fit(
    X_lstm_train,
    y_train,
    validation_data=(X_lstm_val, y_val),
    epochs=30,
    batch_size=32,
    callbacks=training_callbacks,
    verbose=1
)


## 11. CNN Data Preparation

The piano-roll matrices are converted into a NumPy array and given a final channel dimension. This produces the four-dimensional shape expected by a two-dimensional convolutional layer:

`(samples, pitches, time steps, channels)`

In [ ]:
def prepare_cnn_data(matrices, train_indices, val_indices, test_indices):
    """Convert piano-roll matrices into four-dimensional CNN input arrays."""

    # Convert the list of matrices into one float32 array to reduce memory use
    # and match TensorFlow's expected numerical type.
    X = np.asarray(matrices, dtype=np.float32)

    # CNN layers expect [samples, height, width, channels].
    # The piano roll is grayscale/binary, so one channel dimension is added.
    X = np.expand_dims(X, axis=-1)

    # Apply the same split indices used for the LSTM so both models are evaluated
    # on exactly the same musical pieces.
    return X[train_indices], X[val_indices], X[test_indices]


X_cnn_train, X_cnn_val, X_cnn_test = prepare_cnn_data(
    cnn_matrices, train_idx, val_idx, test_idx
)

print("CNN training shape:", X_cnn_train.shape)


## 12. Build the CNN Model

The CNN applies filters across the piano-roll matrices to identify local pitch and timing patterns. Pooling reduces the feature-map size, while batch normalization and dropout help stabilize training and reduce overfitting.

In [ ]:
def build_cnn_model(input_shape, number_of_classes):
    """Build a CNN classifier for piano-roll representations of MIDI files."""

    model = Sequential([
        Input(shape=input_shape),

        # The first convolution learns local pitch-and-time patterns.
        Conv2D(32, kernel_size=(3, 3), activation="relu", padding="same"),

        # Batch normalization stabilizes activation values and can speed training.
        BatchNormalization(),

        # Pooling reduces the matrix dimensions while retaining prominent features.
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.20),

        # Deeper convolutional blocks learn increasingly complex musical patterns.
        Conv2D(64, kernel_size=(3, 3), activation="relu", padding="same"),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.25),

        Conv2D(128, kernel_size=(3, 3), activation="relu", padding="same"),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.30),

        # Flatten converts the learned feature maps into a single vector.
        Flatten(),

        # The dense layer combines extracted features before classification.
        Dense(128, activation="relu"),
        Dropout(0.40),

        # Softmax produces the predicted probability for each composer.
        Dense(number_of_classes, activation="softmax")
    ])

    # Use the same optimizer, loss function, and accuracy metric as the LSTM
    # so the model comparison is as consistent as possible.
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


cnn_model = build_cnn_model(
    input_shape=X_cnn_train.shape[1:],
    number_of_classes=len(class_names)
)

cnn_model.summary()


### 12.1 Train the CNN

The CNN is trained using the same class labels and data split as the LSTM. This makes the final comparison more meaningful.

In [ ]:
# Train the CNN using the same epoch limit, batch size, validation strategy,
# and callbacks used for the LSTM so the comparison remains consistent.
cnn_history = cnn_model.fit(
    X_cnn_train,
    y_train,
    validation_data=(X_cnn_val, y_val),
    epochs=30,
    batch_size=32,
    callbacks=training_callbacks,
    verbose=1
)


## 13. Visualize Training Performance

Training and validation curves help identify whether a model is learning effectively or beginning to overfit. A growing gap between training and validation performance may indicate that the model is memorizing the training data.

In [ ]:
def plot_training_history(history, model_name):
    """Plot training and validation accuracy and loss across epochs."""

    # Convert the Keras history dictionary to a DataFrame for easier plotting.
    history_df = pd.DataFrame(history.history)

    # Compare training and validation accuracy.
    # A widening gap may indicate that the model is overfitting the training data.
    plt.figure(figsize=(8, 5))
    plt.plot(history_df["accuracy"], label="Training Accuracy")
    plt.plot(history_df["val_accuracy"], label="Validation Accuracy")
    plt.title(f"{model_name} Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Compare training and validation loss.
    # Validation loss is also the value monitored by the training callbacks.
    plt.figure(figsize=(8, 5))
    plt.plot(history_df["loss"], label="Training Loss")
    plt.plot(history_df["val_loss"], label="Validation Loss")
    plt.title(f"{model_name} Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    plt.show()


plot_training_history(lstm_history, "LSTM")
plot_training_history(cnn_history, "CNN")


## 14. Model Evaluation

Each model is evaluated using the held-out test set. The evaluation includes:

- Accuracy
- Macro-averaged precision
- Macro-averaged recall
- Macro-averaged F1 score
- Per-composer classification report
- Confusion matrix

Macro averages give each composer equal weight, which is useful when the dataset is not perfectly balanced.

In [ ]:
def evaluate_model(model, X_test, y_test, class_names, model_name):
    """Evaluate a trained model and return summary classification metrics."""

    # Generate one probability distribution per test example.
    probabilities = model.predict(X_test, verbose=0)

    # Select the composer class with the highest predicted probability.
    predictions = np.argmax(probabilities, axis=1)

    # Accuracy reports the overall proportion of correct predictions.
    accuracy = accuracy_score(y_test, predictions)

    # Macro averaging gives every composer equal weight, regardless of class size.
    # This is useful when the dataset may contain different numbers of pieces per composer.
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test,
        predictions,
        average="macro",
        zero_division=0
    )

    print(f"{model_name} Test Accuracy: {accuracy:.4f}")
    print(f"{model_name} Macro Precision: {precision:.4f}")
    print(f"{model_name} Macro Recall: {recall:.4f}")
    print(f"{model_name} Macro F1 Score: {f1:.4f}")
    print()

    # Display class-level precision, recall, and F1 scores to identify
    # which composers are easier or harder for the model to recognize.
    print(classification_report(
        y_test,
        predictions,
        target_names=class_names,
        zero_division=0
    ))

    # Build a confusion matrix to show the specific composer pairs
    # that the model most frequently confuses.
    cm = confusion_matrix(y_test, predictions)

    matrix_display = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names
    )
    matrix_display.plot(xticks_rotation=45)
    plt.title(f"{model_name} Confusion Matrix")
    plt.tight_layout()
    plt.show()

    # Return the summary metrics in dictionary form so both models
    # can be placed in one comparison table.
    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    }


lstm_results = evaluate_model(
    lstm_model, X_lstm_test, y_test, class_names, "LSTM"
)

cnn_results = evaluate_model(
    cnn_model, X_cnn_test, y_test, class_names, "CNN"
)


## 15. Compare the Models

The summary table places the primary evaluation metrics side by side. The best model should not be selected from accuracy alone. Precision, recall, F1 score, confusion matrices, training stability, and class balance should also be considered.

In [ ]:
# Place the LSTM and CNN metrics in one table for direct comparison.
comparison_df = pd.DataFrame([lstm_results, cnn_results])

# Sort by macro F1 score because it balances precision and recall while
# giving each composer equal importance.
comparison_df = comparison_df.sort_values("F1 Score", ascending=False)

# Format the metrics to four decimal places for a clear academic presentation.
display(comparison_df.style.format({
    "Accuracy": "{:.4f}",
    "Precision": "{:.4f}",
    "Recall": "{:.4f}",
    "F1 Score": "{:.4f}"
}))


## 16. Hyperparameter Optimization

The assignment requires model optimization. A limited and well-documented comparison is sufficient. The team can test a small number of reasonable alternatives rather than conducting an exhaustive search.

Potential adjustments include:

- LSTM units: 64, 128, or 256
- CNN filter counts: 32, 64, or 128
- Dropout rates: 0.20 to 0.50
- Batch sizes: 16, 32, or 64
- Learning rates: 0.001 or 0.0005
- Sequence lengths or piano-roll time steps

Record the configurations tested and explain why the final configuration was selected.

In [ ]:
# Create a structured table for recording model-tuning experiments.
# Keeping a written record prevents configurations from being compared only from memory
# and supports the optimization discussion required in the final report.
optimization_results = pd.DataFrame(columns=[
    "Model",
    "Configuration",
    "Validation Accuracy",
    "Validation Loss",
    "Notes"
])

optimization_results


## 17. Results and Discussion

Complete this section after running the final models.

The discussion should address:

- Which model performed better overall?
- Were some composers easier to identify than others?
- Which composers were commonly confused?
- Did the training curves show evidence of overfitting?
- Did class imbalance appear to affect the results?
- How did feature representation influence performance?

**Draft results narrative:**

The [CNN/LSTM] produced the strongest overall test performance, with an accuracy of [value] and a macro F1 score of [value]. The confusion matrices showed that [composer] was identified most consistently, while the model frequently confused [composer] with [composer]. This may reflect similarities in musical style, differences in the number of available training examples, or limitations in the selected feature representation.

The training and validation curves indicated [little/moderate/significant] overfitting. The use of dropout, early stopping, and learning-rate reduction helped control this behavior. Overall, the results suggest that [sequence-based/piano-roll-based] features were more effective for this dataset.

## 18. Conclusion

This project examined whether deep learning could identify classical composers from labeled MIDI files. The workflow included MIDI parsing, data cleaning, feature extraction, model development, training, evaluation, and comparison.

The LSTM treated each composition as a sequence of notes and chords, while the CNN analyzed a piano-roll representation of pitch activity over time. The final results showed that [best model] achieved the strongest performance based on [accuracy/F1 score]. The findings demonstrate that composer-specific patterns can be learned from symbolic music data, although performance was influenced by dataset size, class balance, composition length, and feature design.

Future improvements could include using more training data, incorporating note duration and velocity more explicitly, applying data augmentation, testing bidirectional LSTMs, using attention or transformer architectures, and evaluating models on compositions that were not represented during training.

## 19. Limitations

Important limitations should be documented so the results are not overstated:

- The dataset may contain an unequal number of pieces per composer.
- Pieces from the same collection or arrangement may be highly similar.
- MIDI files may vary in instrumentation, length, and quality.
- Truncation can remove information from longer compositions.
- Piano-roll compression can reduce timing detail.
- A strong test result does not necessarily mean the model will generalize to all classical music.

## 20. References

Update this section with every external source, library, framework, and dataset used. References should follow APA 7 style.

**Initial framework and library references**

Abadi, M., et al. (2016). TensorFlow: A system for large-scale machine learning. *12th USENIX Symposium on Operating Systems Design and Implementation*, 265–283.

Cuthbert, M. S., & Ariza, C. (2010). music21: A toolkit for computer-aided musicology and symbolic music data. *Proceedings of the 11th International Society for Music Information Retrieval Conference*, 637–642.

Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830.

TensorFlow. (n.d.). *TensorFlow documentation*. https://www.tensorflow.org/

**Dataset reference**

[Add the complete APA citation or acknowledgment for the provided composer MIDI dataset.]

**Course materials**

[Add citations or acknowledgments for the course notebooks and instructional materials used.]

## 21. Reproducibility Notes

Before submission, record the following:

- Python version
- TensorFlow version
- Runtime type used in Google Colab
- Dataset location and folder structure
- Number of files included and excluded
- Random seed
- Final hyperparameters
- Number of training epochs
- Final train, validation, and test sizes

These details will make it easier for another person to reproduce the results.

## 22. Submission Checklist

- [ ] All cells run from beginning to end without errors.
- [ ] Dataset paths do not expose private information.
- [ ] Data preprocessing is clearly documented.
- [ ] Feature extraction is clearly documented.
- [ ] Both CNN and LSTM models are included.
- [ ] Model summaries and training curves are shown.
- [ ] Accuracy, precision, recall, F1 score, and confusion matrices are included.
- [ ] Optimization experiments are documented.
- [ ] Results are interpreted rather than only displayed.
- [ ] Conclusion includes findings and future improvements.
- [ ] References are complete and formatted in APA 7 style.
- [ ] Team contributions and AI assistance are acknowledged.
- [ ] Notebook is exported to the required PDF or HTML format.